In [31]:
import sqlite3
import pandas as pd

DB_PATH = "/Users/darraghdonnelly/dev/Database/recovered.db"
TEST_SIZE = 0.20
RANDOM_STATE = 42

with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql_query(
        "SELECT src, sex, age, distance_m, time_s FROM concat_results",
        conn,
    )

In [32]:
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV

# same preprocessing as model_comparison nb

df["distance_bucket"] = (
    df["distance_m"].round().astype(int)
    .map({42195: "42k", 16093: "16k", 5000: "5k"})
)
# convert distnace to log scale
df["log_distance"] = np.log(df["distance_m"].astype(float))

feature_cols = ["log_distance", "age", "sex"]

train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["distance_bucket"],
)

X_train = train_df[feature_cols]
y_train = np.log(train_df["time_s"].astype(float))
train_buckets = train_df["distance_bucket"]


In [33]:
from sklearn.metrics import mean_absolute_error, make_scorer

# retun mae in second for evaluation in grid search
def mae_seconds(y_true_log, y_pred_log):
    return mean_absolute_error(np.exp(y_true_log), np.exp(y_pred_log))

# scorer for grid search uses above func to compute mae in secs 
mae_scorer = make_scorer(mae_seconds, greater_is_better=False)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)


In [34]:
from sklearn.linear_model import Ridge

# grid search for ridge regression
ridge_search = GridSearchCV(
    estimator=Ridge(),
    param_grid={"alpha": [0.1, 0.5, 0.75, 1.0]},
    scoring=mae_scorer,
    cv=cv.split(X_train, train_buckets),
)

ridge_search.fit(X_train, y_train)

# print best params and cv mae 
print("Ridge")
print(ridge_search.best_params_)

Ridge
{'alpha': 0.1}


In [35]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# range of hps to be searched
rf_grid = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [5, 10, 15, 20, None],
    "min_samples_leaf": [1, 2, 4],
}

# random search for RF regressor
rf_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE),
    param_distributions=rf_grid,
    n_iter=10,
    scoring=mae_scorer,
    cv=cv.split(X_train, train_buckets),
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

rf_search.fit(X_train, y_train)

print("Random Forest")
print(rf_search.best_params_)

Random Forest
{'n_estimators': 200, 'min_samples_leaf': 2, 'max_depth': 5}


In [36]:
# range of hps to be searched
gb_grid = {
    "loss": ["squared_error", "absolute_error", "huber"],
    "learning_rate": [0.1, 0.3, 0.5, 0.8],
    "n_estimators": [100, 200, 400],
}

gb_search = RandomizedSearchCV(
    estimator=GradientBoostingRegressor(random_state=RANDOM_STATE),
    param_distributions=gb_grid,
    n_iter=10,
    scoring=mae_scorer,
    cv=cv.split(X_train, train_buckets),
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

gb_search.fit(X_train, y_train)

print("Gradient Boosting")
print(gb_search.best_params_)

Gradient Boosting
{'n_estimators': 400, 'loss': 'huber', 'learning_rate': 0.1}


In [37]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV

xgb_grid = {
        "n_estimators": [100, 200, 300, 400],
        "max_depth": [2, 3, 4, 5, 6],
        "learning_rate": [0.01, 0.05, 0.1, 0.2, 0.3],
        "subsample": [0.7, 0.8, 0.9, 1.0],
        "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
        "min_child_weight": [1, 3, 5],
    }

# objective set to sqrd err for regression
xgb_model = XGBRegressor(objective="reg:squarederror", random_state=RANDOM_STATE)

xgb_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=xgb_grid,
    n_iter=20,
    scoring=mae_scorer,
    cv=cv.split(X_train, train_buckets),
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

xgb_search.fit(X_train, y_train)

print("XGBoost")
print(xgb_search.best_params_)

XGBoost
{'subsample': 0.7, 'n_estimators': 400, 'min_child_weight': 3, 'max_depth': 5, 'learning_rate': 0.01, 'colsample_bytree': 1.0}


In [45]:
print("\nBest CV MAE:")
print("Ridge")
print(round(-ridge_search.best_score_, 2), "seconds")

print("\nRandom Forest")
print(round(-rf_search.best_score_, 2), "seconds")

print("\nGradient Boosting")
print(round(-gb_search.best_score_, 2), "seconds")

print("\nXGBoost")
print(round(-xgb_search.best_score_, 2), "seconds")


Best CV MAE:
Ridge
2197.49 seconds

Random Forest
2173.84 seconds

Gradient Boosting
2171.72 seconds

XGBoost
2171.97 seconds
